In [2]:
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_ROOT = Path("/store/users/xu/2025_MSc_b2b/roamm")

INPUT__PATH = (PROJECT_ROOT / "outputs" / "roamm_words_with_length_and_frequency.csv")

words = pd.read_csv(INPUT__PATH, keep_default_na=False)

words.shape

(10839, 10)

In [3]:
words.head()

,story_name,page,sentence_id,sentence,word_position,words,word_key,normalized_word,word_length,frequency_zipf
0,history_of_film,0,history_of_film_0,"The history of film began in the 1890s, with t...",0,The,4f29e384-84f8-4dc8-bd07-e4d351038c53,The,3,7.73
1,history_of_film,0,history_of_film_0,"The history of film began in the 1890s, with t...",1,history,1d6ed8cf-e2b5-43ce-b190-56f8ec83d14c,history,7,5.39
2,history_of_film,0,history_of_film_0,"The history of film began in the 1890s, with t...",2,of,7e7b8839-ee21-47ac-8a31-9d078a751723,of,2,7.4
3,history_of_film,0,history_of_film_0,"The history of film began in the 1890s, with t...",3,film,e5f11c54-eed2-4a3d-b843-2f75c9632a48,film,4,5.2
4,history_of_film,0,history_of_film_0,"The history of film began in the 1890s, with t...",4,began,db267961-974c-491a-8f3f-74e3f819920f,began,5,5.07


In [4]:
words.groupby("sentence_id")["story_name"].nunique().max()

np.int64(1)

In [5]:
words[
    ["story_name", "sentence_id", "word_position", "words", "sentence"]
].head(30)

,story_name,sentence_id,word_position,words,sentence
0,history_of_film,history_of_film_0,0,The,"The history of film began in the 1890s, with t..."
1,history_of_film,history_of_film_0,1,history,"The history of film began in the 1890s, with t..."
2,history_of_film,history_of_film_0,2,of,"The history of film began in the 1890s, with t..."
3,history_of_film,history_of_film_0,3,film,"The history of film began in the 1890s, with t..."
4,history_of_film,history_of_film_0,4,began,"The history of film began in the 1890s, with t..."
5,history_of_film,history_of_film_0,5,in,"The history of film began in the 1890s, with t..."
6,history_of_film,history_of_film_0,6,the,"The history of film began in the 1890s, with t..."
7,history_of_film,history_of_film_0,7,"1890s,","The history of film began in the 1890s, with t..."
8,history_of_film,history_of_film_0,8,with,"The history of film began in the 1890s, with t..."
9,history_of_film,history_of_film_0,9,the,"The history of film began in the 1890s, with t..."


In [9]:
sentence_id = "pluto_0"

sentence_df = (words[words["sentence_id"] == sentence_id].sort_values("word_position"))
sentence_df[["word_position", "words", "sentence"]]

,word_position,words,sentence
2141,0,Pluto,Pluto (minor-planet designation: 134340 Pluto)...
2142,1,(minor-planet,Pluto (minor-planet designation: 134340 Pluto)...
2143,2,designation:,Pluto (minor-planet designation: 134340 Pluto)...
2144,3,134340,Pluto (minor-planet designation: 134340 Pluto)...
2145,4,Pluto),Pluto (minor-planet designation: 134340 Pluto)...
2146,5,is,Pluto (minor-planet designation: 134340 Pluto)...
2147,6,a,Pluto (minor-planet designation: 134340 Pluto)...
2148,7,dwarf,Pluto (minor-planet designation: 134340 Pluto)...
2149,8,planet,Pluto (minor-planet designation: 134340 Pluto)...
2150,9,in,Pluto (minor-planet designation: 134340 Pluto)...


In [10]:
original_words = sentence_df["words"].tolist()

reconstructed = " ".join(original_words)
original_sentence = sentence_df["sentence"].iloc[0]

print("Original:")
print(original_sentence)

print("\nReconstructed:")
print(reconstructed)

print("\nSame:")
print(original_sentence == reconstructed)

Original:
Pluto (minor-planet designation: 134340 Pluto) is a dwarf planet in the Kuiper belt, a ring of bodies beyond Neptune.

Reconstructed:
Pluto (minor-planet designation: 134340 Pluto) is a dwarf planet in the Kuiper belt, a ring of bodies beyond Neptune.

Same:
True


In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "gpt2",
    use_fast=True
)

encoded = tokenizer(
    reconstructed,
    return_offsets_mapping=True,
    add_special_tokens=False
)

tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"]
)

offsets = encoded["offset_mapping"]

for token, offset in zip(tokens, offsets):
    print(token, offset)

Pl (0, 2)
uto (2, 5)
Ġ( (5, 7)
min (7, 10)
or (10, 12)
- (12, 13)
planet (13, 19)
Ġdesignation (19, 31)
: (31, 32)
Ġ13 (32, 35)
43 (35, 37)
40 (37, 39)
ĠPluto (39, 45)
) (45, 46)
Ġis (46, 49)
Ġa (49, 51)
Ġdwarf (51, 57)
Ġplanet (57, 64)
Ġin (64, 67)
Ġthe (67, 71)
ĠK (71, 73)
ui (73, 75)
per (75, 78)
Ġbelt (78, 83)
, (83, 84)
Ġa (84, 86)
Ġring (86, 91)
Ġof (91, 94)
Ġbodies (94, 101)
Ġbeyond (101, 108)
ĠNeptune (108, 116)
. (116, 117)


In [12]:
text = ""
word_spans = []

for word in original_words:
    if text:
        text += " "

    start = len(text)
    text += word
    end = len(text)

    word_spans.append((start, end))

In [13]:
for word, span in zip(original_words, word_spans):
    print(word, span)

Pluto (0, 5)
(minor-planet (6, 19)
designation: (20, 32)
134340 (33, 39)
Pluto) (40, 46)
is (47, 49)
a (50, 51)
dwarf (52, 57)
planet (58, 64)
in (65, 67)
the (68, 71)
Kuiper (72, 78)
belt, (79, 84)
a (85, 86)
ring (87, 91)
of (92, 94)
bodies (95, 101)
beyond (102, 108)
Neptune. (109, 117)


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

file = Path(
    "/scratch/data/ROAMM/derivatives/synced/"
    "sub-10014/"
    "sub-10014_task-ReMind_run-01_mldata.pkl"
)

df = pd.read_pickle(file)

df.shape

(168704, 139)

In [3]:
fix = df.loc[
    df["first_pass_reading"].eq(True)
    & df["fix_R_tStart"].notna()
].copy()

fix[
    ["time", "fix_R_tStart", "fix_R_fixed_word"]
].head()

,time,fix_R_tStart,fix_R_fixed_word
3215,12.558594,12.220262,NaN
3216,12.562500,12.220262,NaN
3217,12.566406,12.220262,NaN
3218,12.570312,12.220262,NaN
3219,12.574219,12.220262,NaN


In [4]:
fix_events = (
    fix
    .drop_duplicates("fix_R_tStart")
    .copy()
)

fix_events[
    ["time", "fix_R_tStart", "fix_R_fixed_word"]
].head()

,time,fix_R_tStart,fix_R_fixed_word
3215,12.558594,12.220262,NaN
3316,12.953125,12.949263,volume.
3393,13.253906,13.253262,volume.
3437,13.425781,13.424263,(minor-planet
3479,13.589844,13.589262,Pluto


In [5]:
fix_events["eeg_sample"] = fix_events.index

In [6]:
fix_events["sync_error_ms"] = (
    fix_events["time"] - fix_events["fix_R_tStart"]
) * 1000

In [7]:
fix_events[
    [
        "eeg_sample",
        "time",
        "fix_R_tStart",
        "sync_error_ms",
        "fix_R_fixed_word",
    ]
].head(20)

,eeg_sample,time,fix_R_tStart,sync_error_ms,fix_R_fixed_word
3215,3215,12.558594,12.220262,338.33125,NaN
3316,3316,12.953125,12.949263,3.86250,volume.
3393,3393,13.253906,13.253262,0.64375,volume.
3437,3437,13.425781,13.424263,1.51875,(minor-planet
3479,3479,13.589844,13.589262,0.58125,Pluto
3522,3522,13.757812,13.757263,0.55000,Pluto
3580,3580,13.984375,13.983263,1.11250,(minor-planet
3635,3635,14.199219,14.196262,2.95625,(minor-planet
3699,3699,14.449219,14.447262,1.95625,designation:
3749,3749,14.644531,14.644262,0.26875,134340


In [8]:
fix_events["sync_error_ms"].abs().max()

np.float64(914.4875000000638)

In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

pkl_path = Path(
    "/scratch/data/ROAMM/derivatives/synced/"
    "sub-10111/sub-10111_task-ReMind_run-05_mldata.pkl"
)

df = pd.read_pickle(pkl_path)

In [22]:
list(df.columns[:64])

['Fp1',
 'AF7',
 'AF3',
 'F1',
 'F3',
 'F5',
 'F7',
 'FT7',
 'FC5',
 'FC3',
 'FC1',
 'C1',
 'C3',
 'C5',
 'T7',
 'TP7',
 'CP5',
 'CP3',
 'CP1',
 'P1',
 'P3',
 'P5',
 'P7',
 'P9',
 'PO7',
 'PO3',
 'O1',
 'Iz',
 'Oz',
 'POz',
 'Pz',
 'CPz',
 'Fpz',
 'Fp2',
 'AF8',
 'AF4',
 'Afz',
 'Fz',
 'F2',
 'F4',
 'F6',
 'F8',
 'FT8',
 'FC6',
 'FC4',
 'FC2',
 'FCz',
 'Cz',
 'C2',
 'C4',
 'C6',
 'T8',
 'TP8',
 'CP6',
 'CP4',
 'CP2',
 'P2',
 'P4',
 'P6',
 'P8',
 'P10',
 'PO8',
 'PO4',
 'O2']

In [23]:
EEG_CHANNELS = [
    "Fp1", "AF7", "AF3", "F1", "F3", "F5", "F7", "FT7",
    "FC5", "FC3", "FC1", "C1", "C3", "C5", "T7", "TP7",
    "CP5", "CP3", "CP1", "P1", "P3", "P5", "P7", "P9",
    "PO7", "PO3", "O1", "Iz", "Oz", "POz", "Pz", "CPz",
    "Fpz", "Fp2", "AF8", "AF4", "Afz", "Fz", "F2", "F4",
    "F6", "F8", "FT8", "FC6", "FC4", "FC2", "FCz", "Cz",
    "C2", "C4", "C6", "T8", "TP8", "CP6", "CP4", "CP2",
    "P2", "P4", "P6", "P8", "P10", "PO8", "PO4", "O2",
]

In [24]:
assert list(df.columns[:64]) == EEG_CHANNELS

In [25]:
eeg = np.load(
    "/scratch/data/ROAMM/outputs/eeg/sub-10111_run5_eeg.npy"
)

eeg.shape

(64, 198912)

In [26]:
for i, ch in enumerate(EEG_CHANNELS):
    assert np.allclose(
        eeg[i],
        df[ch].to_numpy()
    )

print("All EEG channels match.")

All EEG channels match.
